# Field Shift · Punto 2: comparar rotaciones (modelo B)

**Qué ve el productor** (pantalla "Compare rotations" del `INFO.md`): para su condado, el rango de rinde esperado de maíz o soja según la rotación, probado con el clima de cada año histórico.

**Qué hace:** un modelo de ML estima cómo el clima y el suelo se asocian con el rinde por condado (`rend_bu_ac`), y el efecto de la rotación se mide aparte con efectos fijos (paso 14). No recomienda: muestra rangos y el productor decide.

Es el **modelo B (escenarios de rotación)** del `CLAUDE.md` de esta carpeta: solo usa lo que un escenario puede fijar (clima de un año histórico, suelo, rotación y tendencia). **Nada de NDVI ni EVI del año**: son consecuencia del cultivo, no algo que se pueda elegir.

**Datos:** la parte `condado/` de la entrega `data/fieldshift_datos_v1.2/` (en la raíz del repo, fuera de Git; cómo conseguirla en `notebooks/ml/README.md`). Si todavía no está, la notebook corre igual y cada paso dice "en espera".

**Cómo usarla:**

1. Elegí el cultivo en el paso 1 (`CULTIVO = "maiz"` o `"soja"`).
2. Corré todo de arriba hacia abajo (Kernel → Restart & Run All).
3. Repetí con el otro cultivo. Las salidas de los dos se juntan en los mismos archivos.

## 0. Librerías

- **pandas / numpy:** tablas y cuentas.
- **scikit-learn:** el modelo lineal (Ridge), la preparación de variables y las métricas.
- **LightGBM:** muchos árboles de decisión chiquitos, cada uno corrigiendo al anterior. Si no está instalado, se usa uno equivalente de scikit-learn.
- **joblib:** guarda el modelo entrenado en un archivo.

In [ ]:
import datetime
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

try:
    import lightgbm as lgb
    USAR_LGBM = True
except ImportError:
    from sklearn.ensemble import HistGradientBoostingRegressor
    USAR_LGBM = False

SEMILLA = 0
print("LightGBM disponible:", USAR_LGBM)

## 1. Parámetros

Todo lo que se puede cambiar está en esta celda.

- **Años:** desde 2009, porque las transiciones de rotación del CDL arrancan ahí (2008 está vacío). 2024 y 2025 quedan como **prueba final**: no se miran hasta el final (`MIRAR_PRUEBA`).
- **Features:** elegidas por criterio agronómico (sección 5 del `CLAUDE.md`). Para la rotación se usa una sola variable: qué % del cultivo se sembró sobre el mismo cultivo del año anterior (maíz sobre maíz, soja sobre soja). Las de la tabla mensual se piden como `(variable, mes)` y se convierten en columnas como `deficit_mm_07`.
- Si alguna feature no viene en la entrega, el paso 4 avisa y sigue sin ella. No se inventa.

In [ ]:
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
VERSION_DATOS = "fieldshift_datos_v1.2"   # cada entrega nueva viene en su propia carpeta
DATOS = RAIZ / "data" / VERSION_DATOS / "condado"
SALIDA = RAIZ / "data" / "ml"
CARPETA_MODELOS = RAIZ / "modelos"

CULTIVO = "maiz"            # "maiz" o "soja": se corre una vez por cultivo
TARGET = "rend_bu_ac"
ANIO_DESDE = 2009
ANIOS_PRUEBA = [2024, 2025]
MIRAR_PRUEBA = False        # pasalo a True una sola vez, al final
NOMBRES_ESTADO = {"17": "Illinois", "18": "Indiana", "19": "Iowa"}

# Tabla anual
FEATURES_ANUAL = {
    "maiz": ["prec_jul_mm", "tmax_jul_c", "dias_tmax_gt30_jul"],          # floración
    "soja": ["prec_ago_mm"],                                             # llenado de grano
    "comunes": ["prec_temporada_mm", "gdd_temporada", "deficit_temporada_mm"],
}
# Tabla mensual: (variable, mes)
FEATURES_MENSUAL = {
    "maiz": [("dias_tmax_gt35", 7), ("deficit_mm", 7)],
    "soja": [("tmax_c", 8), ("deficit_mm", 8)],
    "comunes": [("prec_power_mm", 4), ("prec_power_mm", 5),              # exceso de agua en primavera
                ("dias_lluvia_gt50", 4), ("dias_lluvia_gt50", 5),
                ("gwettop", 4), ("gwettop", 5), ("gwetroot", 4), ("gwetroot", 5)],
}
SUELO_NUM = ["aws0_100_mm", f"nccpi_{CULTIVO}", "soc0_30_gm2", "pendiente_media_pct", "twi_medio"]
SUELO_CAT = ["clase_drenaje", "grupo_hidrologico"]
# Rotación: % del cultivo sembrado sobre el MISMO cultivo del año anterior (maíz sobre maíz, soja sobre soja).
# La otra columna ("después de otro cultivo") no se usa: dentro de un condado es casi su espejo
# (correlación -0,91 en maíz) y con las dos juntas el modelo no separa los efectos.
ROTACION = {"maiz": ["pct_maiz_tras_maiz"], "soja": ["pct_soja_tras_soja"]}[CULTIVO]

HAY_DATOS = (DATOS / "tabla_anual.csv").exists()
print("Cultivo:", CULTIVO, "| ¿Está la entrega de datos?", HAY_DATOS)
if not HAY_DATOS:
    print(f"Falta {DATOS}. La notebook queda en espera hasta que esté.")

## 2. Cargar los datos

`fips` se lee como texto para no perder el cero de la izquierda. El diccionario dice qué columnas se pueden usar como entrada.

In [ ]:
if HAY_DATOS:
    anual = pd.read_csv(DATOS / "tabla_anual.csv", dtype={"fips": str})
    mensual = pd.read_csv(DATOS / "tabla_mensual.csv", dtype={"fips": str})
    ruta_dic = DATOS / "diccionario.csv"
    dic = pd.read_csv(ruta_dic) if ruta_dic.exists() else None
    print("tabla_anual:", anual.shape, "| tabla_mensual:", mensual.shape,
          "| diccionario:", None if dic is None else dic.shape)
else:
    print("En espera de los datos.")

## 3. Mirar los datos antes de modelar

Cuántos condados tienen rinde de este cultivo en cada año y estado. Fijate que 2024 y 2025 tienen menos condados.

In [ ]:
if HAY_DATOS:
    con_rinde = anual[(anual["cultivo"] == CULTIVO) & anual[TARGET].notna()]
    display(pd.crosstab(con_rinde["anio"], con_rinde["fips"].str[:2].map(NOMBRES_ESTADO), margins=True))
else:
    print("En espera de los datos.")

## 4. Armar la tabla de modelado

Una fila por condado y año, con el rinde y las features:

1. La tabla mensual se da vuelta para que cada mes sea una columna (`deficit_mm_07` = déficit de julio).
2. Se queda con el cultivo elegido, desde 2009, solo filas con rinde.
3. Avisa qué features pedidas no vinieron, y cuáles no figuran como `entrada` en el diccionario.
4. **Separa la rotación en dos partes.** Con datos reales se vio que los condados que hacen más maíz continuo son los más productivos (por suelo y ubicación), no que el maíz continuo rinda más. Si el modelo ve el % de rotación crudo, mezcla las dos cosas. Por eso cada `pct_*` se parte en:
   - `<pct>_media`: cómo rota **normalmente** ese condado (su promedio en los años de entrenamiento del fold);
   - `<pct>_desvio`: cuánto se apartó **ese año** de lo normal.

   Así el efecto de rotar sale de comparar a cada condado consigo mismo. El promedio se calcula dentro del fold (paso 7).

In [ ]:
def armar_tabla(anual, mensual, dic):
    pedidas_m = FEATURES_MENSUAL[CULTIVO] + FEATURES_MENSUAL["comunes"]
    faltan = sorted({v for v, _ in pedidas_m if v not in mensual.columns})
    vars_m = sorted({v for v, _ in pedidas_m if v in mensual.columns})
    meses = sorted({m for _, m in pedidas_m})

    m = mensual.assign(m=mensual["mes"].str[5:7].astype(int))
    piv = m[m["m"].isin(meses)].pivot_table(index=["fips", "anio"], columns="m", values=vars_m)
    piv.columns = [f"{c}_{mes:02d}" for c, mes in piv.columns]
    cols_m = [f"{v}_{mes:02d}" for v, mes in pedidas_m if f"{v}_{mes:02d}" in piv.columns]

    t = anual[(anual["cultivo"] == CULTIVO) & anual[TARGET].notna() & (anual["anio"] >= ANIO_DESDE)]
    t = t.merge(piv[cols_m].reset_index(), on=["fips", "anio"], how="left", validate="one_to_one")

    pedidas_num = FEATURES_ANUAL[CULTIVO] + FEATURES_ANUAL["comunes"] + SUELO_NUM + ROTACION
    faltan += [c for c in pedidas_num + SUELO_CAT if c not in t.columns]
    num = [c for c in pedidas_num if c in t.columns] + cols_m
    cat = [c for c in SUELO_CAT if c in t.columns]

    if dic is not None and {"archivo", "columna", "rol"} <= set(dic.columns):
        roles = dic.set_index(["archivo", "columna"])["rol"]
        clave = {c: ("tabla_mensual.csv", c.rsplit("_", 1)[0]) if c in cols_m else ("tabla_anual.csv", c)
                 for c in num + cat}
        raras = [c for c, k in clave.items() if k in roles.index and roles[k] != "entrada"]
        if raras:
            print("OJO: no figuran como 'entrada' en el diccionario:", raras)

    t = t.assign(estado=t["fips"].str[:2]).reset_index(drop=True)
    return t, num, cat, faltan


def separar_rotacion(datos, medias):
    """Agrega <pct>_media (cómo rota normalmente el condado) y <pct>_desvio (cuánto se apartó ese año)."""
    d = datos.copy()
    for c in ROTACION:
        media = d["fips"].map(medias[c])
        # Condado que no estaba en el train (por ejemplo, con un estado afuera): se usa su propio
        # historial de rotación. Es una entrada que se conoce, no el rinde.
        media = media.fillna(d.groupby("fips")[c].transform("mean"))
        d[f"{c}_media"] = media
        d[f"{c}_desvio"] = d[c] - media
    return d


if HAY_DATOS:
    tabla, NUM_CRUDAS, CAT, faltan = armar_tabla(anual, mensual, dic)
    ROTACION = [c for c in ROTACION if c in NUM_CRUDAS]
    NUM = [c for c in NUM_CRUDAS if c not in ROTACION] + [f"{c}_{p}" for c in ROTACION for p in ("media", "desvio")]
    FEATURES = NUM + CAT
    print(f"Tabla de modelado: {len(tabla)} filas, {len(FEATURES)} features para el modelo")
    if faltan:
        print("No vienen en la entrega (se sigue sin ellas; anotalo en docs/ml/pedidos_datos.md):", faltan)
    display(tabla[NUM_CRUDAS + CAT].isna().mean().mul(100).round(1).to_frame("% vacías").T)
else:
    print("En espera de los datos.")

## 5. Folds: cómo se separan los datos

- **Desarrollo (2009–2023):** se deja **un año afuera por vez**. El modelo aprende con los demás años y se evalúa en el que quedó afuera. Así simula predecir un año que no vio.
- **Prueba final (2024–2025):** no se toca hasta el final.

No se mezclan filas al azar: todos los condados de un mismo año comparten el clima, y el modelo haría trampa.

Los folds se guardan en `data/ml/folds.csv` para que cualquiera pueda repetir la validación.

In [ ]:
def guardar_por_cultivo(df, ruta):
    """Guarda un CSV con columna 'cultivo', reemplazando solo las filas del cultivo actual."""
    ruta.parent.mkdir(parents=True, exist_ok=True)
    if ruta.exists():
        viejo = pd.read_csv(ruta, dtype={"fips": str})
        df = pd.concat([viejo[viejo["cultivo"] != CULTIVO], df], ignore_index=True)
    df.to_csv(ruta, index=False)


if HAY_DATOS:
    dev = tabla[~tabla["anio"].isin(ANIOS_PRUEBA)]
    prueba = tabla[tabla["anio"].isin(ANIOS_PRUEBA)]

    folds = tabla[["fips", "anio", "estado"]].assign(cultivo=CULTIVO)
    folds["bloque"] = np.where(folds["anio"].isin(ANIOS_PRUEBA), "prueba", "desarrollo")
    folds["fold_anio"] = folds["anio"].where(folds["bloque"] == "desarrollo").astype("Int64")
    folds["fold_estado"] = folds["estado"].where(folds["bloque"] == "desarrollo")
    guardar_por_cultivo(folds, SALIDA / "folds.csv")

    print(f"Desarrollo: {len(dev)} filas, años {dev['anio'].min()}–{dev['anio'].max()} "
          f"({dev['anio'].nunique()} folds) | Prueba: {len(prueba)} filas")
else:
    print("En espera de los datos.")

## 6. Tendencia y baselines

**Tendencia:** el rinde sube año a año por la tecnología. El modelo no aprende el rinde directo, sino **cuánto se aparta de la tendencia** (la anomalía). La tendencia es una recta por estado, ajustada **solo con los años de entrenamiento del fold**: si usara el año que quedó afuera, sería trampa.

**Baselines:** reglas simples que el modelo tiene que superar. Si no les gana, no aporta.

- **Media del condado:** el promedio de ese condado en los años de entrenamiento.
- **Tendencia del condado:** una recta propia del condado (si tiene al menos 5 años; si no, la del estado).
- **Año anterior:** el rinde del condado el año anterior.

Cuando a un baseline le falta el dato, se usa la tendencia del estado.

In [ ]:
def ajustar_tendencia(tr):
    """Recta rinde ~ año por estado, solo con el train. Devuelve los coeficientes."""
    por_estado = {e: np.polyfit(g["anio"], g[TARGET], 1) for e, g in tr.groupby("estado")}
    return {"por_estado": por_estado, "global": np.polyfit(tr["anio"], tr[TARGET], 1)}


def aplicar_tendencia(coefs, datos, anio=None):
    """Valor de la tendencia para cada fila. Con 'anio' fijo, usa ese año para todas."""
    anios = datos["anio"] if anio is None else pd.Series(anio, index=datos.index)
    return np.array([np.polyval(coefs["por_estado"].get(e, coefs["global"]), a)
                     for e, a in zip(datos["estado"], anios)])


def calcular_baselines(tr, te):
    tend_estado = aplicar_tendencia(ajustar_tendencia(tr), te)
    media = te["fips"].map(tr.groupby("fips")[TARGET].mean()).to_numpy()

    rectas = {f: np.polyfit(g["anio"], g[TARGET], 1) for f, g in tr.groupby("fips") if len(g) >= 5}
    tend_condado = np.array([np.polyval(rectas[f], a) if f in rectas else np.nan
                             for f, a in zip(te["fips"], te["anio"])])

    rinde_todos = anual[anual["cultivo"] == CULTIVO].set_index(["fips", "anio"])[TARGET]
    anterior = np.array([rinde_todos.get((f, a - 1), np.nan) for f, a in zip(te["fips"], te["anio"])])

    b = pd.DataFrame({"media_condado": media, "tendencia_condado": tend_condado,
                      "anio_anterior": anterior}, index=te.index)
    return b.apply(lambda col: col.fillna(pd.Series(tend_estado, index=te.index)))


BASELINES = ["media_condado", "tendencia_condado", "anio_anterior"]

## 7. Los modelos

En este orden (sección 5 del `CLAUDE.md`):

1. **Ridge:** una regresión lineal con freno, para que no exagere con pocas observaciones. Completa vacíos con la mediana y pone las variables en la misma escala.
2. **LightGBM poco profundo:** árboles de 3 niveles con al menos 40 filas por hoja, para que no memorice.

Los dos aprenden la **anomalía** (rinde − tendencia), y al predecir se le suma la tendencia de nuevo. La tendencia y el promedio de rotación de cada condado se calculan **solo con el train del fold**, y se guardan junto con el modelo en un "ajuste".

In [ ]:
MODELOS_A_PROBAR = ["ridge", "lightgbm"]


def crear_modelo(nombre):
    if nombre == "ridge":
        prep = ColumnTransformer([
            ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUM),
            ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                                  OneHotEncoder(handle_unknown="ignore")), CAT),
        ])
        return Pipeline([("prep", prep), ("modelo", Ridge(alpha=10.0))])

    prep = ColumnTransformer([
        ("num", "passthrough", NUM),
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1,
                               encoded_missing_value=-1), CAT),
    ])
    if USAR_LGBM:
        arboles = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.03, max_depth=3, num_leaves=7,
                                    min_child_samples=40, subsample=0.8, subsample_freq=1,
                                    colsample_bytree=0.8, random_state=SEMILLA, verbose=-1)
    else:
        arboles = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.03, max_depth=3,
                                                min_samples_leaf=40, random_state=SEMILLA)
    return Pipeline([("prep", prep), ("modelo", arboles)])


def entrenar(nombre, tr):
    """Ajusta tendencia, promedios de rotación y modelo solo con 'tr'. Devuelve un dict 'ajuste'."""
    ajuste = {"tendencia": ajustar_tendencia(tr), "medias": tr.groupby("fips")[ROTACION].mean()}
    x = separar_rotacion(tr, ajuste["medias"])[FEATURES]
    ajuste["modelo"] = crear_modelo(nombre).fit(x, tr[TARGET] - aplicar_tendencia(ajuste["tendencia"], tr))
    return ajuste


def predecir(ajuste, datos, anio=None):
    x = separar_rotacion(datos, ajuste["medias"])[FEATURES]
    return ajuste["modelo"].predict(x) + aplicar_tendencia(ajuste["tendencia"], datos, anio)

## 8. Validación: dejar un año afuera

Para cada año de 2009 a 2023: se entrena con los demás y se predice ese año, con los baselines y los dos modelos. Al final, cada fila tiene una predicción hecha **sin haber visto su año** ("fuera del fold").

De paso se mide la **importancia por permutación**: se mezcla una variable al azar y se ve cuánto empeora el modelo. Si empeora mucho, esa variable importa.

In [ ]:
if HAY_DATOS:
    oof = dev[["fips", "anio", "estado", TARGET]].copy()
    importancias = []
    for anio in sorted(dev["anio"].unique()):
        tr, te = dev[dev["anio"] != anio], dev[dev["anio"] == anio]
        b = calcular_baselines(tr, te)
        oof.loc[te.index, BASELINES] = b.to_numpy()
        for nombre in MODELOS_A_PROBAR:
            ajuste = entrenar(nombre, tr)
            oof.loc[te.index, nombre] = predecir(ajuste, te)
            if nombre == "lightgbm":
                x_te = separar_rotacion(te, ajuste["medias"])[FEATURES]
                pi = permutation_importance(ajuste["modelo"], x_te, te[TARGET] - aplicar_tendencia(ajuste["tendencia"], te),
                                            scoring="neg_mean_absolute_error", n_repeats=3,
                                            random_state=SEMILLA)
                importancias.append(pd.Series(pi.importances_mean, index=FEATURES, name=anio))
    print("Validación lista:", dev["anio"].nunique(), "años dejados afuera de a uno.")
else:
    print("En espera de los datos.")

## 9. Resultados: ¿le gana a los baselines?

- **MAE:** error promedio en bu/ac (más bajo es mejor).
- **RMSE:** parecido, pero castiga más los errores grandes.
- **R²:** qué parte de la variación explica (1 es perfecto, 0 es como no saber nada).

Se muestran **por estado** y **por año**, no solo el promedio: un buen promedio puede esconder un año o un estado donde anda mal (por ejemplo, la sequía de 2012).

In [ ]:
PREDICTORES = BASELINES + MODELOS_A_PROBAR


def metricas(df):
    return pd.DataFrame({c: {"MAE": mean_absolute_error(df[TARGET], df[c]),
                             "RMSE": root_mean_squared_error(df[TARGET], df[c]),
                             "R2": r2_score(df[TARGET], df[c])} for c in PREDICTORES}).T


if HAY_DATOS:
    print("MAE por estado (bu/ac):")
    mae_estado = {NOMBRES_ESTADO.get(e, e): metricas(g)["MAE"] for e, g in oof.groupby("estado")}
    mae_estado["TOTAL"] = metricas(oof)["MAE"]
    display(pd.DataFrame(mae_estado).round(1))

    print("Todas las métricas, total:")
    display(metricas(oof).round(2))

    mae_anio = pd.DataFrame({a: metricas(g)["MAE"] for a, g in oof.groupby("anio")}).T
    print("MAE por año (fold):")
    display(mae_anio.round(1))

    ELEGIDO = min(MODELOS_A_PROBAR, key=lambda c: mean_absolute_error(oof[TARGET], oof[c]))
    MEJOR_BASELINE = min(BASELINES, key=lambda c: mean_absolute_error(oof[TARGET], oof[c]))
    mae_elegido = mean_absolute_error(oof[TARGET], oof[ELEGIDO])
    mae_base = mean_absolute_error(oof[TARGET], oof[MEJOR_BASELINE])
    print(f"Modelo elegido: {ELEGIDO} (MAE {mae_elegido:.1f}) | mejor baseline: {MEJOR_BASELINE} (MAE {mae_base:.1f})")
    print(f"Le gana en {(mae_anio[ELEGIDO] < mae_anio[MEJOR_BASELINE]).sum()} de {len(mae_anio)} años.")
    if mae_elegido >= mae_base:
        print("ATENCIÓN: el modelo no le gana al mejor baseline. Revisá features y datos antes de seguir.")
else:
    print("En espera de los datos.")

## 10. ¿Sirve en un estado que no vio?

Se entrena con dos estados y se evalúa en el tercero. Es la prueba de que el modelo podría usarse en zonas nuevas. Se compara contra la tendencia de los otros estados, porque las reglas del condado no existen para un estado sin historia.

Ojo: acá los mismos años están en train y en evaluación, así que mide solo si generaliza en el espacio, no en el tiempo.

In [ ]:
if HAY_DATOS and dev["estado"].nunique() > 1:
    filas = []
    for e in sorted(dev["estado"].unique()):
        tr, te = dev[dev["estado"] != e], dev[dev["estado"] == e]
        ajuste = entrenar(ELEGIDO, tr)
        filas.append({"estado_afuera": NOMBRES_ESTADO.get(e, e),
                      "MAE_modelo": mean_absolute_error(te[TARGET], predecir(ajuste, te)),
                      "MAE_tendencia_otros_estados": mean_absolute_error(te[TARGET], aplicar_tendencia(ajuste["tendencia"], te))})
    display(pd.DataFrame(filas).round(1))
else:
    print("En espera de los datos.")

## 11. ¿Qué variables pesan?

Importancia por permutación del LightGBM en cada año dejado afuera. Se mira:

- **media:** cuánto empeora el MAE al mezclar esa variable (más alto = más importante);
- **% de folds en que ayuda:** si una variable solo sirve en uno o dos años, probablemente es ruido.

Es una **asociación, no una causa**. Para el pitch: "en estos datos, los julios más calurosos se asocian a menos rinde", nunca "porque".

In [ ]:
if HAY_DATOS and importancias:
    imp = pd.concat(importancias, axis=1)
    resumen_imp = pd.DataFrame({"media": imp.mean(axis=1),
                                "% folds en que ayuda": (imp > 0).mean(axis=1) * 100})
    display(resumen_imp.sort_values("media", ascending=False).round(2))
else:
    print("En espera de los datos.")

## 12. Banda de incertidumbre (p10–p90)

Con los errores fuera del fold del modelo elegido se calcula cuánto suele errar hacia abajo y hacia arriba. La banda `[pred + q10, pred + q90]` debería contener al **80 %** de los casos.

En el total da 80 % por construcción; lo que importa es si se mantiene **por estado** y, al final, en la prueba 2024–2025.

In [ ]:
if HAY_DATOS:
    residuos = oof[TARGET] - oof[ELEGIDO]
    Q10, Q90 = np.quantile(residuos, [0.1, 0.9])
    dentro = residuos.between(Q10, Q90)
    print(f"Banda: pred {Q10:+.1f} a pred {Q90:+.1f} bu/ac")
    cobertura = dentro.groupby(oof["estado"].map(NOMBRES_ESTADO)).mean().mul(100).round(1)
    cobertura["TOTAL"] = round(dentro.mean() * 100, 1)
    display(cobertura.to_frame("% dentro de la banda (objetivo 80)"))
else:
    print("En espera de los datos.")

## 13. Modelo final y prueba 2024–2025

El modelo final se entrena con **todos** los años de desarrollo y se guarda en `modelos/` (fuera de Git) para que lo usen el punto 3 y el punto 4. También se guarda la tabla de modelado en `data/ml/tabla_modelado.csv`, así esos puntos no tienen que rearmarla.

La prueba final está apagada. Cambiá `MIRAR_PRUEBA` a `True` en el paso 1 **una sola vez**, cuando ya no vayas a cambiar nada. Si después de verla seguís ajustando, deja de ser un examen honesto.

In [ ]:
if HAY_DATOS:
    ajuste_final = entrenar(ELEGIDO, dev)
    CARPETA_MODELOS.mkdir(parents=True, exist_ok=True)
    ruta_modelo = CARPETA_MODELOS / f"punto2_rotaciones_{CULTIVO}.joblib"
    # Para usarlo en otra notebook: aplicar separar_rotacion(datos, medias) antes de modelo.predict
    # y sumar la tendencia (ver predecir en el paso 7).
    joblib.dump({**ajuste_final, "features": FEATURES, "rotacion": ROTACION, "cultivo": CULTIVO,
                 "target": TARGET, "q10": Q10, "q90": Q90, "version_datos": VERSION_DATOS}, ruta_modelo)
    print("Modelo guardado en", ruta_modelo.relative_to(RAIZ))
    columnas_tabla = ["fips", "anio", "estado", "cultivo", TARGET] + NUM_CRUDAS + CAT
    guardar_por_cultivo(tabla[columnas_tabla], SALIDA / "tabla_modelado.csv")
    print("Tabla de modelado guardada en data/ml/tabla_modelado.csv")

    if MIRAR_PRUEBA:
        pred = predecir(ajuste_final, prueba)
        res = prueba[["fips", "anio", "estado", TARGET]].assign(**{ELEGIDO: pred})
        res[BASELINES] = calcular_baselines(dev, prueba).to_numpy()
        PREDICTORES_PRUEBA = BASELINES + [ELEGIDO]
        tabla_prueba = pd.DataFrame({NOMBRES_ESTADO.get(e, e): {c: mean_absolute_error(g[TARGET], g[c])
                                     for c in PREDICTORES_PRUEBA} for e, g in res.groupby("estado")})
        tabla_prueba["TOTAL"] = {c: mean_absolute_error(res[TARGET], res[c]) for c in PREDICTORES_PRUEBA}
        print("MAE en la prueba 2024–2025 (bu/ac):")
        display(tabla_prueba.round(1))
        r = res[TARGET] - res[ELEGIDO]
        print(f"Cobertura de la banda p10–p90 en la prueba: {r.between(Q10, Q90).mean() * 100:.1f} % (objetivo 80)")
    else:
        print("Prueba 2024–2025 sin mirar todavía (MIRAR_PRUEBA = False).")
else:
    print("En espera de los datos.")

## 14. Efecto de la rotación (efectos fijos)

El modelo de ML es bueno para el clima y el suelo, pero **no logra medir el efecto de la rotación**: con datos por condado, "más maíz sobre maíz" aparece junto con otras cosas (condados más productivos, años de precios altos). Por eso el efecto de rotar se mide aparte, con un método estándar en economía agrícola: **efectos fijos**.

La idea es comparar cada condado **consigo mismo** y contra lo que pasó **ese mismo año en su estado**:

- **Mismo condado:** saca todo lo fijo del condado (suelo, ubicación, si es bueno o malo).
- **Mismo estado y año:** saca lo que afecta a todos a la vez ese año (precios, tecnología, una sequía general).
- **Clima local:** saca lo que le tocó a ese condado ese año (calor, lluvia, déficit).

Lo que queda responde: *"en los años en que este condado hizo más del cultivo sobre sí mismo que lo normal, comparado con sus vecinos ese año, ¿rindió más o menos?"*

La tabla muestra cómo cambia el resultado a medida que se suman controles. El que se usa es el último, el más estricto. Los errores se agrupan por condado (un condado se parece a sí mismo entre años). Solo usa los años de desarrollo: la prueba 2024–2025 sigue sin tocar.

In [ ]:
from math import erf, sqrt


def estimar_efecto_rotacion(datos, efectos, controles):
    """Regresión del rinde contra 'rot' con efectos fijos y errores agrupados por condado."""
    d = datos.dropna(subset=["rot"] + controles).reset_index(drop=True)
    partes = [d[["rot"] + controles].to_numpy(float)]
    partes += [pd.get_dummies(d[e], drop_first=True).to_numpy(float) for e in efectos]
    X = np.column_stack(partes + [np.ones(len(d))])
    y = d[TARGET].to_numpy(float)
    inv = np.linalg.pinv(X.T @ X)
    b = inv @ X.T @ y
    u = y - X @ b
    centro = np.zeros((X.shape[1], X.shape[1]))
    for idx in d.groupby("fips").indices.values():
        s = X[idx].T @ u[idx]
        centro += np.outer(s, s)
    g, (n, k) = d["fips"].nunique(), X.shape
    se = np.sqrt((inv @ centro @ inv)[0, 0] * g / (g - 1) * (n - 1) / (n - k))
    p = 2 * (1 - 0.5 * (1 + erf(abs(b[0] / se) / sqrt(2))))
    return {"beta": b[0], "ic_bajo": b[0] - 1.96 * se, "ic_alto": b[0] + 1.96 * se, "p": p, "n": n}


if HAY_DATOS:
    mismo = ROTACION[0]
    CLIMA_FE = [c for c in NUM if c not in SUELO_NUM and not c.startswith("pct_")]
    d_fe = dev.assign(rot=dev[mismo], estado_anio=dev["estado"] + "_" + dev["anio"].astype(str))
    rinde_medio = d_fe[TARGET].mean()

    especificaciones = {
        "Sin controles": ([], []),
        "Mismo condado": (["fips"], []),
        "Mismo condado + mismo año": (["fips", "anio"], []),
        "Mismo condado + mismo año + clima": (["fips", "anio"], CLIMA_FE),
        "Mismo condado + mismo estado y año + clima": (["fips", "estado_anio"], CLIMA_FE),
    }
    filas = []
    for nombre, (efectos, controles) in especificaciones.items():
        r = estimar_efecto_rotacion(d_fe, efectos, controles)
        filas.append({"controles": nombre, "bu/ac por punto": r["beta"], "IC95 bajo": r["ic_bajo"],
                      "IC95 alto": r["ic_alto"], "p": r["p"],
                      "100 % vs 0 % (% del rinde)": r["beta"] * 100 / rinde_medio * 100})
    display(pd.DataFrame(filas).set_index("controles").round(3))

    EFECTO_ROTACION = r   # el último: el más estricto
    sin_2012 = estimar_efecto_rotacion(d_fe[d_fe["anio"] != 2012], ["fips", "estado_anio"], CLIMA_FE)
    guardar_por_cultivo(pd.DataFrame([{
        "cultivo": CULTIVO, "columna": mismo, "bu_ac_por_punto": round(r["beta"], 4),
        "ic95_bajo": round(r["ic_bajo"], 4), "ic95_alto": round(r["ic_alto"], 4), "p": r["p"], "n": r["n"],
        "rinde_medio_bu_ac": round(rinde_medio, 1),
        "continuo_vs_rotado_pct": round(r["beta"] * 100 / rinde_medio * 100, 1),
        "version_datos": VERSION_DATOS}]), SALIDA / "efecto_rotacion.csv")
    print(f"Efecto usado: {EFECTO_ROTACION['beta']:+.2f} bu/ac por punto de {mismo} "
          f"(IC95 {EFECTO_ROTACION['ic_bajo']:+.2f} a {EFECTO_ROTACION['ic_alto']:+.2f}). "
          f"Sin 2012: {sin_2012['beta']:+.2f}.")
else:
    print("En espera de los datos.")

## 15. Escenarios de rotación (la salida para la app)

Para cada condado se arma el rinde de tres opciones, con **el clima de cada año histórico** (2009–2023) y el suelo del condado:

| `rotacion` | Qué es | Cómo se calcula |
|---|---|---|
| `actual` | Seguir rotando como rota normalmente el condado | Modelo de ML (paso 13) |
| `rotado` | Nada del cultivo sobre sí mismo (maíz siempre después de soja) | `actual` + efecto de la rotación (paso 14) × (0 % − % normal del condado) |
| `continuo` | Todo el cultivo sobre sí mismo (maíz sobre maíz) | `actual` + efecto de la rotación × (100 % − % normal del condado) |

Así **cada parte sale de donde es confiable**: el rango según el clima y el suelo, del modelo de ML; la diferencia entre rotaciones, de los efectos fijos.

**Supuesto:** el efecto se midió con cambios chicos en el % del condado (unos 6 puntos) y se extiende de forma proporcional a 0 % y 100 %. El paso 16 compara el resultado con lo que miden los experimentos de campo (maíz: 11–28 % menos).

La banda `p10`–`p90` suma dos incertidumbres: el error del modelo de ML (paso 12) y el intervalo de confianza del efecto de la rotación. La tendencia se fija en `ANIO_TENDENCIA`. Los ids de rotación son **provisorios**: falta acordarlos con el front (`INFO.md`, sección 4.6). Salen dos archivos:

- `data/ml/escenarios_rotacion.csv`: `fips, rotacion, cultivo, anio_clima, pred, p10, p90`. Una fila por clima.
- `data/ml/escenarios_rotacion_resumen.csv`: por condado y rotación, `mediana_clima, p10_clima, p90_clima`: cómo varía el rinde **entre climas** (año normal, malo y bueno).

In [ ]:
ANIO_TENDENCIA = max(ANIOS_PRUEBA)
PCT_ROTACION = {"actual": None, "rotado": 0.0, "continuo": 100.0}

if HAY_DATOS:
    base = dev.copy()
    normal = ajuste_final["medias"][mismo].loc[base["fips"]].to_numpy()
    base[mismo] = normal                       # rotación normal del condado: desvío del año = 0
    pred_actual = predecir(ajuste_final, base, anio=ANIO_TENDENCIA)
    b, b_bajo, b_alto = EFECTO_ROTACION["beta"], EFECTO_ROTACION["ic_bajo"], EFECTO_ROTACION["ic_alto"]

    partes = []
    for rotacion, pct in PCT_ROTACION.items():
        cambio = np.zeros(len(base)) if pct is None else pct - normal
        pred = pred_actual + b * cambio
        extra_bajo = np.minimum((b_bajo - b) * cambio, (b_alto - b) * cambio)
        extra_alto = np.maximum((b_bajo - b) * cambio, (b_alto - b) * cambio)
        partes.append(pd.DataFrame({"fips": base["fips"].to_numpy(), "rotacion": rotacion, "cultivo": CULTIVO,
                                    "anio_clima": base["anio"].to_numpy(), "pred": pred,
                                    "p10": pred + Q10 + extra_bajo, "p90": pred + Q90 + extra_alto}))
    escenarios = pd.concat(partes, ignore_index=True).round(1)

    resumen_esc = (escenarios.groupby(["fips", "rotacion", "cultivo"])["pred"]
                   .agg(mediana_clima="median",
                        p10_clima=lambda s: s.quantile(0.1),
                        p90_clima=lambda s: s.quantile(0.9))
                   .round(1).reset_index())

    guardar_por_cultivo(escenarios, SALIDA / "escenarios_rotacion.csv")
    guardar_por_cultivo(resumen_esc, SALIDA / "escenarios_rotacion_resumen.csv")
    print(f"Guardados {len(escenarios)} escenarios en data/ml/. Ejemplo, primer condado:")
    display(resumen_esc[resumen_esc["fips"] == resumen_esc["fips"].iloc[0]])
else:
    print("En espera de los datos.")

## 16. Chequeo agronómico

Efecto documentado: el maíz continuo rinde **11–28 % menos** que el maíz en rotación (Iowa State, 10 años en 7 sitios; ver `ML_NOTAS_CDL.md`, sección 6), y la soja también rinde menos después de soja. El chequeo pide:

1. que el efecto de la rotación (paso 14) sea negativo y su intervalo de confianza no toque el cero;
2. en maíz, que la diferencia entre 100 % y 0 % caiga cerca del rango de los experimentos;
3. que `rotado` rinda más que `continuo` en los escenarios.

**Historia:** las primeras versiones medían la rotación con el modelo de ML y **fallaban este chequeo en maíz**: daban que el maíz continuo rinde más, porque los condados que más lo hacen son los más productivos y porque en los años de precios altos se siembra más maíz. Con efectos fijos (paso 14) se corrigió.

In [ ]:
RANGO_LITERATURA = {"maiz": (11, 28)}   # % menos de rinde del cultivo continuo; para soja no hay un rango citado en el repo

if HAY_DATOS:
    pct_efecto = -EFECTO_ROTACION["beta"] * 100 / rinde_medio * 100
    ok_signo = EFECTO_ROTACION["ic_alto"] < 0
    print(f"1. Efecto de rotación: {EFECTO_ROTACION['beta']:+.2f} bu/ac por punto, IC95 "
          f"[{EFECTO_ROTACION['ic_bajo']:+.2f}, {EFECTO_ROTACION['ic_alto']:+.2f}] -> {'OK' if ok_signo else 'FALLA'}")
    if CULTIVO in RANGO_LITERATURA:
        lo, hi = RANGO_LITERATURA[CULTIVO]
        print(f"2. El cultivo continuo rinde {pct_efecto:.0f} % menos (experimentos: {lo}–{hi} %) -> "
              f"{'OK' if lo * 0.5 <= pct_efecto <= hi * 1.5 else 'REVISAR'}")
    ancho = resumen_esc.pivot(index="fips", columns="rotacion", values="mediana_clima")
    dif = ancho["rotado"] - ancho["continuo"]
    print(f"3. Rotado − continuo: mediana {dif.median():+.1f} bu/ac; rotado rinde más en {(dif > 0).mean() * 100:.0f} % de los condados")
    if not ok_signo or dif.median() <= 0:
        print("ATENCIÓN: no se reproduce el beneficio de rotar. No mostrar esta comparación en la app.")
else:
    print("En espera de los datos.")

## 17. Registrar el experimento

Cada corrida suma una fila a `data/ml/experimentos.csv` con la fecha, el cultivo, el modelo, las features, las métricas y el efecto de la rotación. Así se puede comparar qué cambió entre corridas.

In [ ]:
if HAY_DATOS:
    fila = {
        "fecha": datetime.datetime.now().isoformat(timespec="seconds"),
        "version_datos": VERSION_DATOS,
        "cultivo": CULTIVO, "target": TARGET, "modelo": ELEGIDO,
        "features": ";".join(FEATURES),
        "folds": f"dejar_un_anio_afuera_{dev['anio'].min()}-{dev['anio'].max()}",
        "mae_modelo": round(mae_elegido, 2),
        "mejor_baseline": MEJOR_BASELINE, "mae_mejor_baseline": round(mae_base, 2),
        "mae_por_estado": json.dumps({k: round(v[ELEGIDO], 2) for k, v in mae_estado.items()}),
        "mae_por_anio": json.dumps({int(a): round(v, 2) for a, v in mae_anio[ELEGIDO].items()}),
        "q10": round(Q10, 2), "q90": round(Q90, 2),
        "efecto_rotacion_bu_ac_por_punto": round(EFECTO_ROTACION["beta"], 3),
        "efecto_rotacion_ic95": f"{EFECTO_ROTACION['ic_bajo']:.3f};{EFECTO_ROTACION['ic_alto']:.3f}",
        "miro_prueba": MIRAR_PRUEBA,
    }
    ruta_exp = SALIDA / "experimentos.csv"
    ruta_exp.parent.mkdir(parents=True, exist_ok=True)
    if ruta_exp.exists():
        previo = pd.read_csv(ruta_exp)
        pd.concat([previo, pd.DataFrame([fila])], ignore_index=True).to_csv(ruta_exp, index=False)
    else:
        pd.DataFrame([fila]).to_csv(ruta_exp, index=False)
    print("Experimento registrado en", ruta_exp.relative_to(RAIZ))
else:
    print("En espera de los datos.")

## Próximos pasos

- [ ] Anotar en `docs/ml/pedidos_datos.md` las features que el paso 4 diga que faltan.
- [ ] Sacar los pares de features con |r| > 0,9, quedándose con la más fácil de explicar.
- [ ] Early stopping del LightGBM y ajuste de `alpha` del Ridge con un bloque de años reservado dentro del desarrollo (nunca con la prueba).
- [ ] Sumar las prácticas del censo (`extras/practicas_censo.csv`, el último censo ≤ año).
- [ ] Punto 3: explicaciones en `punto3_explicaciones.ipynb` (todo lo necesario en `PUNTO2.md`, sección "Para el punto 3").
- [ ] Cuando llegue NLDAS (v2), repetir el paso 14 con su humedad del suelo como control extra, para confirmar el efecto.
- [ ] Cuando llegue `economia` (v2), pasar los escenarios a US$/acre (rinde × precio − costos) en un paso aparte, sin meter precios en el modelo.
- [ ] Comprobar el efecto de la rotación a escala de lote con `lotes/` (NDVI por píxel según el cultivo anterior).
- [ ] Acordar con el front los ids de `rotacion` y generar `data/ml/front/escenarios_rotacion.json` (`INFO.md`, sección 4.6).
- [ ] Escribir `docs/ml/model_card.md` (modelo B, métricas contra los baselines por fold, limitaciones).
- [ ] Al final, una sola vez: `MIRAR_PRUEBA = True`.